![Plotly](./img/plotly.png)

# Visualización interactiva con Plotly Express

[Plotly](https://plotly.com/python/) es una librería de visualización que genera gráficos **interactivos**: puedes pasar el ratón por encima de los puntos para ver sus valores (*hover*), hacer zoom, desplazarte y ocultar o mostrar series pinchando en la leyenda.

Plotly tiene dos interfaces:
* **`plotly.express` (`px`)**: la interfaz de alto nivel. Con una sola función y un DataFrame obtenemos un gráfico completo, de forma muy parecida a Seaborn. Es la que usaremos en este notebook.
* **`plotly.graph_objects` (`go`)**: la interfaz de bajo nivel, con la que se construye el gráfico pieza a pieza. Es más laboriosa, pero da control total. Al final veremos cómo combinarla con `px`.

Contenido:
1. Carga y limpieza de datos
2. Gráficos de líneas
3. Gráficos de dispersión
4. Gráficos de barras
5. Gráficos de tarta y donut
6. Gráficos de burbujas
7. Histogramas
8. Diagramas de caja
9. Matriz de dispersión
10. Subplots con *facets*
11. Dispersión 3D
12. Mapas
13. Animaciones
14. Personalización y `graph_objects`
15. Exportar a HTML

In [1]:
# %pip install plotly
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# Cargamos la librería JavaScript de Plotly desde internet en lugar de incrustarla
# en cada gráfico: así el notebook ocupa mucho menos
pio.renderers.default = "plotly_mimetype+notebook_connected"

## 1. Carga y limpieza de datos

Usaremos el ranking mundial de universidades de *Times Higher Education* (2011-2016). Sus columnas son:

* `world_rank`: posición en el ranking
* `university_name`: nombre de la universidad
* `country`: país
* `teaching`: puntuación en docencia
* `international`: puntuación en internacionalización
* `research`: puntuación en investigación
* `citations`: puntuación en citas
* `income`: puntuación en ingresos por transferencia a la industria
* `total_score`: puntuación total
* `num_students`: número de estudiantes
* `student_staff_ratio`: estudiantes por profesor
* `international_students`: porcentaje de estudiantes internacionales
* `female_male_ratio`: proporción mujeres : hombres
* `year`: año del ranking

In [2]:
times_df = pd.read_csv("./data/timesData.csv")
times_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2603 entries, 0 to 2602
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   world_rank              2603 non-null   object 
 1   university_name         2603 non-null   object 
 2   country                 2603 non-null   object 
 3   teaching                2603 non-null   float64
 4   international           2603 non-null   object 
 5   research                2603 non-null   float64
 6   citations               2603 non-null   float64
 7   income                  2603 non-null   object 
 8   total_score             2603 non-null   object 
 9   num_students            2544 non-null   object 
 10  student_staff_ratio     2544 non-null   float64
 11  international_students  2536 non-null   object 
 12  female_male_ratio       2370 non-null   object 
 13  year                    2603 non-null   int64  
dtypes: float64(4), int64(1), object(9)
memor

In [3]:
times_df.head()

,world_rank,university_name,country,teaching,international,research,citations,income,total_score,num_students,student_staff_ratio,international_students,female_male_ratio,year
0,1,Harvard University,United States of America,99.7,72.4,98.7,98.8,34.5,96.1,"20,152",8.9,25%,NaN,2011
1,2,California Institute of Technology,United States of America,97.7,54.6,98.0,99.9,83.7,96.0,"2,243",6.9,27%,33 : 67,2011
2,3,Massachusetts Institute of Technology,United States of America,97.8,82.3,91.4,99.9,87.5,95.6,"11,074",9.0,33%,37 : 63,2011
3,4,Stanford University,United States of America,98.3,29.5,98.1,99.2,64.3,94.3,"15,596",7.8,22%,42 : 58,2011
4,5,Princeton University,United States of America,90.9,70.3,95.4,99.9,-,94.2,"7,929",8.4,27%,45 : 55,2011


Varias columnas numéricas se han leído como texto (`object`): el ranking incluye valores como `=12` (empate) o `201-225` (tramo), las puntuaciones usan `-` cuando no hay dato, el número de estudiantes lleva separador de miles y el porcentaje de internacionales lleva `%`. Las convertimos a número; lo que no se pueda convertir queda como `NaN`:

In [4]:
times_df["world_rank"] = pd.to_numeric(times_df["world_rank"].str.replace("=", ""), errors="coerce")
for col in ["international", "income", "total_score"]:
    times_df[col] = pd.to_numeric(times_df[col], errors="coerce")
times_df["num_students"] = pd.to_numeric(times_df["num_students"].str.replace(",", ""), errors="coerce")
times_df["international_students"] = pd.to_numeric(times_df["international_students"].str.replace("%", ""), errors="coerce")

times_df.dtypes

world_rank                float64
university_name            object
country                    object
teaching                  float64
international             float64
research                  float64
citations                 float64
income                    float64
total_score               float64
num_students              float64
student_staff_ratio       float64
international_students    float64
female_male_ratio          object
year                        int64
dtype: object

Prepararemos también un DataFrame con las 100 primeras universidades de 2016, que usaremos en varios ejemplos:

In [5]:
top100_2016 = times_df[(times_df["year"] == 2016) & (times_df["world_rank"] <= 100)]
top100_2016.shape

(100, 14)

## 2. Gráficos de líneas

[Documentación](https://plotly.com/python/line-charts/)

Puntuación en citas y en docencia frente a la posición en el ranking de las 100 primeras universidades de 2016. Pasando una lista de columnas a `y` obtenemos una línea por columna, y con `hover_name` indicamos qué se muestra al pasar el ratón:

In [6]:
fig = px.line(top100_2016,
              x="world_rank",
              y=["citations", "teaching"],
              hover_name="university_name",
              markers=True,
              title="Citas y docencia frente a posición en el ranking (top 100, 2016)",
              labels={"world_rank": "Posición en el ranking", "value": "Puntuación", "variable": "Indicador"})
fig.show()

## 3. Gráficos de dispersión

[Documentación](https://plotly.com/python/line-and-scatter/)

Citas frente a posición de las 100 primeras universidades en 2014, 2015 y 2016. Con `color` separamos los años:

In [7]:
top100_14_16 = times_df[times_df["year"].isin([2014, 2015, 2016]) & (times_df["world_rank"] <= 100)]

fig = px.scatter(top100_14_16,
                 x="world_rank",
                 y="citations",
                 color=top100_14_16["year"].astype(str),
                 hover_name="university_name",
                 title="Citas frente a posición en el ranking (top 100)",
                 labels={"world_rank": "Posición en el ranking", "citations": "Citas", "color": "Año"})
fig.show()

## 4. Gráficos de barras

[Documentación](https://plotly.com/python/bar-charts/)

Citas y docencia de las 3 primeras universidades de 2014. Con `barmode="group"` las barras se colocan una al lado de otra:

In [8]:
top3_2014 = times_df[(times_df["year"] == 2014) & (times_df["world_rank"] <= 3)]

fig = px.bar(top3_2014,
             x="university_name",
             y=["citations", "teaching"],
             barmode="group",
             hover_data=["country"],
             title="Citas y docencia de las 3 primeras universidades (2014)",
             labels={"university_name": "Universidad", "value": "Puntuación", "variable": "Indicador"})
fig.show()

Con `barmode="relative"` (o `"stack"`) las barras se apilan:

In [9]:
fig = px.bar(top3_2014,
             x="university_name",
             y=["citations", "teaching"],
             barmode="relative",
             title="Citas y docencia apiladas (2014)",
             labels={"university_name": "Universidad", "value": "Puntuación", "variable": "Indicador"})
fig.show()

## 5. Gráficos de tarta y donut

[Documentación](https://plotly.com/python/pie-charts/)

Número de estudiantes de las 7 primeras universidades de 2016. El parámetro `hole` convierte la tarta en un donut:

In [10]:
top7_2016 = times_df[(times_df["year"] == 2016) & (times_df["world_rank"] <= 7)]

fig = px.pie(top7_2016,
             values="num_students",
             names="university_name",
             hole=0.3,
             title="Número de estudiantes de las 7 primeras universidades (2016)")
fig.show()

## 6. Gráficos de burbujas

[Documentación](https://plotly.com/python/bubble-charts/)

Posición en el ranking frente a puntuación en docencia de las 20 primeras universidades de 2016, con el tamaño según el número de estudiantes y el color según la puntuación de internacionalización:

In [11]:
top20_2016 = times_df[(times_df["year"] == 2016) & (times_df["world_rank"] <= 20)]

fig = px.scatter(top20_2016,
                 x="world_rank",
                 y="teaching",
                 size="num_students",
                 color="international",
                 hover_name="university_name",
                 size_max=50,
                 title="Docencia frente a posición (top 20, 2016)",
                 labels={"world_rank": "Posición en el ranking", "teaching": "Docencia",
                         "international": "Internacionalización", "num_students": "Estudiantes"})
fig.show()

## 7. Histogramas

[Documentación](https://plotly.com/python/histograms/)

Distribución del número de estudiantes por profesor en 2011, 2012 y 2013. Con `barmode="overlay"` y `opacity` superponemos los histogramas:

In [12]:
ratio_11_13 = times_df[times_df["year"].isin([2011, 2012, 2013])]

fig = px.histogram(ratio_11_13,
                   x="student_staff_ratio",
                   color=ratio_11_13["year"].astype(str),
                   barmode="overlay",
                   opacity=0.6,
                   title="Estudiantes por profesor (2011-2013)",
                   labels={"student_staff_ratio": "Estudiantes por profesor", "color": "Año"})
fig.show()

## 8. Diagramas de caja

[Documentación](https://plotly.com/python/box-plots/)

Recordemos que la caja va del primer cuartil (Q1, percentil 25) al tercero (Q3, percentil 75), la línea central es la mediana y los bigotes llegan hasta 1.5 veces el rango intercuartílico (IQR = Q3 - Q1). Los puntos fuera de los bigotes son posibles *outliers*. Al pasar el ratón, Plotly nos muestra todos estos valores.

Puntuación total y de investigación en 2015:

In [13]:
df_2015 = times_df[times_df["year"] == 2015]

fig = px.box(df_2015,
             y=["total_score", "research"],
             title="Puntuación total y de investigación (2015)",
             labels={"variable": "Indicador", "value": "Puntuación"})
fig.show()

## 9. Matriz de dispersión

[Documentación](https://plotly.com/python/splom/)

Igual que el `pairplot` de Seaborn, nos ayuda a ver la relación entre más de dos variables a la vez:

In [14]:
fig = px.scatter_matrix(df_2015,
                        dimensions=["research", "international", "total_score"],
                        hover_name="university_name",
                        height=700,
                        title="Matriz de dispersión (2015)")
fig.show()

## 10. Subplots con *facets*

[Documentación](https://plotly.com/python/facet-plots/)

En Plotly Express los subplots se crean con `facet_col` y `facet_row`, igual que `col` y `row` en los `catplot`/`relplot` de Seaborn. Para pintar varias variables en paneles distintos primero pasamos los datos a formato largo con `melt`:

In [15]:
df_2015_largo = df_2015.melt(id_vars=["world_rank", "university_name"],
                             value_vars=["research", "citations", "income", "total_score"],
                             var_name="indicador",
                             value_name="puntuacion")

fig = px.scatter(df_2015_largo,
                 x="world_rank",
                 y="puntuacion",
                 facet_col="indicador",
                 facet_col_wrap=2,
                 hover_name="university_name",
                 height=600,
                 title="Investigación, citas, ingresos y puntuación total frente a posición (2015)")
fig.show()

## 11. Dispersión 3D

[Documentación](https://plotly.com/python/3d-scatter-plots/)

A veces dos dimensiones no son suficientes. Con `scatter_3d` añadimos una tercera, y con el color una cuarta. Puedes rotar el gráfico con el ratón:

In [16]:
fig = px.scatter_3d(df_2015,
                    x="world_rank",
                    y="research",
                    z="citations",
                    color="total_score",
                    hover_name="university_name",
                    height=600,
                    title="Posición, investigación y citas (2015)")
fig.update_traces(marker_size=4)
fig.show()

## 12. Mapas

[Documentación](https://plotly.com/python/maps/)

Plotly trae algunos datasets de ejemplo en `px.data`. Uno de ellos es *gapminder*, con la población, esperanza de vida y PIB per cápita de cada país. Con `scatter_geo` situamos cada país en el mapa mediante su código ISO:

In [17]:
gapminder = px.data.gapminder()
gapminder_2007 = gapminder[gapminder["year"] == 2007]

fig = px.scatter_geo(gapminder_2007,
                     locations="iso_alpha",
                     color="continent",
                     hover_name="country",
                     size="pop",
                     projection="natural earth",
                     title="Población por país (2007)")
fig.show()

Y con `choropleth` coloreamos cada país según una variable numérica:

In [18]:
fig = px.choropleth(gapminder_2007,
                    locations="iso_alpha",
                    color="lifeExp",
                    hover_name="country",
                    color_continuous_scale="Viridis",
                    title="Esperanza de vida por país (2007)",
                    labels={"lifeExp": "Esperanza de vida"})
fig.show()

## 13. Animaciones

[Documentación](https://plotly.com/python/animations/)

Con `animation_frame` cada valor de la columna indicada se convierte en un fotograma. El clásico gráfico de Hans Rosling: PIB per cápita frente a esperanza de vida, de 1952 a 2007:

In [19]:
fig = px.scatter(gapminder,
                 x="gdpPercap",
                 y="lifeExp",
                 size="pop",
                 color="continent",
                 hover_name="country",
                 animation_frame="year",
                 animation_group="country",
                 log_x=True,
                 size_max=55,
                 range_x=[100, 100000],
                 range_y=[25, 90],
                 labels={"gdpPercap": "PIB per cápita", "lifeExp": "Esperanza de vida"})
fig.show()

## 14. Personalización y `graph_objects`

Todas las funciones de `px` devuelven un objeto `go.Figure`, así que podemos retocarlo con sus métodos:
* `fig.update_layout(...)`: título, tamaño, leyenda, fuentes, plantilla de estilo (`template`)...
* `fig.update_traces(...)`: aspecto de los elementos dibujados (marcadores, líneas...)
* `fig.update_xaxes(...)` / `fig.update_yaxes(...)`: ejes
* `fig.add_trace(go....)`: añadir elementos construidos con `graph_objects`

Por ejemplo, partimos del gráfico de dispersión del apartado 3, cambiamos el estilo, añadimos una línea con la media de citas y anotamos la primera universidad:

In [20]:
fig = px.scatter(top100_2016,
                 x="world_rank",
                 y="citations",
                 hover_name="university_name",
                 labels={"world_rank": "Posición en el ranking", "citations": "Citas"})

# Línea con la media construida con graph_objects
media = top100_2016["citations"].mean()
fig.add_trace(go.Scatter(x=[1, 100], y=[media, media],
                         mode="lines", name="Media",
                         line=dict(color="red", dash="dash")))

# Anotación
primera = top100_2016.iloc[0]
fig.add_annotation(x=primera["world_rank"], y=primera["citations"],
                   text=primera["university_name"], showarrow=True, arrowhead=2)

fig.update_traces(marker=dict(size=9, color="steelblue"), selector=dict(mode="markers"))
fig.update_layout(template="plotly_white",
                  title="Citas de las 100 primeras universidades (2016)",
                  width=900, height=500)
fig.show()

## 15. Exportar a HTML

Los gráficos de Plotly son interactivos cuando se ven en un navegador. Podemos exportarlos a imagen estática (PNG, JPEG, SVG o PDF, con `fig.write_image`, que necesita la librería `kaleido`) o a un fichero HTML que se abre en cualquier navegador y conserva la interactividad. Con `include_plotlyjs="cdn"` el fichero no incluye la librería Plotly sino que la descarga al abrirse, así que ocupa mucho menos (pero necesita conexión a internet):

In [21]:
# include_plotlyjs="cdn" hace que el HTML cargue Plotly desde internet (fichero mucho más ligero)
fig.write_html("grafico_citas.html", include_plotlyjs="cdn")